In [4]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
def merge_scale_metrics_pipeline():
    GRID_SIZES = [100, 250, 500, 750, 1000]
    PROCESSED_DIR = Path("../../data/processed/").resolve()
    
    output_summary_dir = Path("../../data/unified_scale_matrices").resolve()
    output_summary_dir.mkdir(exist_ok=True)

    target_city_folders = [
        f for f in PROCESSED_DIR.iterdir() 
        if f.is_dir() and f.name != "unified_scale_matrices" and any(char.isdigit() for char in f.name)
    ]
    
    print(f"Verified {len(target_city_folders)} autonomous city repositories for fusion.")

    for size in GRID_SIZES:
        size_metrics_files = []
        for city_folder in target_city_folders:
            metrics_path = city_folder / f"metrics_{size}m.parquet"
            if metrics_path.exists():
                size_metrics_files.append(metrics_path)
        
        if size_metrics_files:
            print(f"\nFusing {len(size_metrics_files)} city blocks for resolution: {size}m...")
            
            raw_merged_df = pd.concat(
                [pd.read_parquet(file) for file in size_metrics_files], 
                ignore_index=True
            )
            
            # =================================================================
            # 🔬 科学修复：真假空值分层防御策略
            # =================================================================
            initial_row_count = len(raw_merged_df)
            
            # 1. 甄别需要填 0 的人造物理空值列
            # 包含：所有景观指数列（如 PLAND_, PD_ 等）以及建筑高度列
            zero_fill_cols = [
                col for col in raw_merged_df.columns 
                if any(keyword in col for keyword in ['_cls_', 'SHDI', 'CONTAG', 'isa_fraction', 'building_height_mean', 'BCR'])
            ]
            
            # 物理规整：没有这种斑块或建筑 = 统计特征为 0
            raw_merged_df[zero_fill_cols] = raw_merged_df[zero_fill_cols].fillna(0.0)
            
            # 2. 识别核心遥感因变量列（LST）
            # 地表温度在城市内部不可能为 0 度，它的 NaN 才是必须剔除的“真遥感盲区”
            critical_core_cols = [col for col in raw_merged_df.columns if 'lst_mean' in col]
            
            # 仅针对 LST 核心列执行严格的行剔除
            clean_merged_df = raw_merged_df.dropna(subset=critical_core_cols).reset_index(drop=True)
            
            # =================================================================
            
            # 存储落盘
            output_parquet_path = output_summary_dir / f"merged_metrics_{size}m.parquet"
            clean_merged_df.to_parquet(output_parquet_path, index=False, engine='pyarrow')
            
            dropped_rows = initial_row_count - len(clean_merged_df)
            print(f"💾 SUCCESS: Matrix committed -> {output_parquet_path.name}")
            print(f"   Final Shape: {clean_merged_df.shape[0]} cells x {clean_merged_df.shape[1]} variables.")
            print(f"   Defensive Clean: Safely kept {len(clean_merged_df)} rows. Purged {dropped_rows} edge-boundary cells with TRUE LST NoData gaps.")
        else:
            print(f"⚠️ No source metrics files detected for grid scale {size}m. Skipping.")

In [6]:
merge_scale_metrics_pipeline()

Verified 99 autonomous city repositories for fusion.

Fusing 99 city blocks for resolution: 100m...
💾 SUCCESS: Matrix committed -> merged_metrics_100m.parquet
   Final Shape: 851633 cells x 48 variables.
   Defensive Clean: Safely kept 851633 rows. Purged 4137 edge-boundary cells with TRUE LST NoData gaps.

Fusing 99 city blocks for resolution: 250m...
💾 SUCCESS: Matrix committed -> merged_metrics_250m.parquet
   Final Shape: 142083 cells x 48 variables.
   Defensive Clean: Safely kept 142083 rows. Purged 422 edge-boundary cells with TRUE LST NoData gaps.

Fusing 99 city blocks for resolution: 500m...
💾 SUCCESS: Matrix committed -> merged_metrics_500m.parquet
   Final Shape: 37715 cells x 48 variables.
   Defensive Clean: Safely kept 37715 rows. Purged 52 edge-boundary cells with TRUE LST NoData gaps.

Fusing 99 city blocks for resolution: 750m...
💾 SUCCESS: Matrix committed -> merged_metrics_750m.parquet
   Final Shape: 17895 cells x 48 variables.
   Defensive Clean: Safely kept 17895

In [7]:
# merge city level data to calculate suhi and add climate label
city_level_data = pd.read_csv("../../data/Unified_City_Rural_Level_Stats_DE.csv")

In [9]:
city_level_data.columns

Index(['system:index', 'CZ_mean', 'CZ_median', 'CZ_stdDev', 'GC_CNT_GAD',
       'GC_DEV_USR', 'GC_DEV_WIG', 'GC_POP_TOT', 'GC_UCA_KM2', 'GC_UCN_MAI',
       'GE_BIO_A01', 'GE_BIO_A02', 'GE_BIO_A03', 'GE_BIO_A04', 'GE_BIO_A05',
       'GE_BIO_A06', 'GE_BIO_A07', 'GE_BIO_A08', 'GE_BIO_A09', 'GE_BIO_A10',
       'GE_BIO_A11', 'GE_BIO_A12', 'GE_BIO_A13', 'GE_BIO_A14', 'GE_BIO_A15',
       'GE_BIO_A16', 'GE_BIO_A17', 'GE_BIO_A18', 'GE_BIO_A19', 'GE_BIO_A20',
       'GE_BIO_A21', 'GE_BIO_A22', 'GE_BIO_A23', 'GE_BIO_A24', 'GE_ECO_CLA',
       'GE_ELV_AVG', 'GE_MRB_LIS', 'GE_MRB_MAI', 'GE_SCL_LST', 'GE_SCL_MPC',
       'ID_UC_G0', 'LST_Rural_mean', 'LST_Rural_median', 'LST_Rural_stdDev',
       'LST_Urban_mean', 'LST_Urban_median', 'LST_Urban_stdDev', 'fid',
       '.geo'],
      dtype='str')

In [10]:
city_level_data_subset = city_level_data[['CZ_median', 'ID_UC_G0', 'LST_Rural_mean']].copy()
city_level_data_subset.describe()

,CZ_median,ID_UC_G0,LST_Rural_mean
count,97.000000,97.000000,97.000000
mean,19.536082,3636.123711,31.265594
std,5.443004,1489.093455,1.924363
min,15.000000,67.000000,26.998214
25%,15.000000,2628.000000,29.807631
50%,15.000000,3980.000000,31.194913
75%,26.000000,4885.000000,32.646782
max,26.000000,5556.000000,35.503284


In [14]:
def integrate_city_intersection_pipeline(city_level_csv_path: str):
    """
    Safely fuses multi-scale grid matrices with city-level baselines by resolving 
    non-aligned city tracking sets via strict 'inner' join intersection.
    Logs missing intersection pairs for scientific methodology reporting.
    """
    GRID_SIZES = [100, 250, 500, 750, 1000]
    PROCESSED_DIR = Path("../../data/processed/").resolve()
    UNIFIED_DIR = PROCESSED_DIR / "unified_scale_matrices"
    
    # 1. Load and Standardize City-level Base DataFrame
    print(f"Loading macro city-level matrix: {Path(city_level_csv_path).name}")
    city_base_df = pd.read_csv(city_level_csv_path)
    
    city_key = 'ID_UC_G0'
    target_background_features = [city_key, 'CZ_median', 'LST_Rural_mean']
    city_base_filtered = city_base_df[target_background_features].copy()
    
    # Standardize primary key format to text to avoid silent datatype drops
    city_base_filtered[city_key] = city_base_filtered[city_key].astype(str).str.strip()
    city_table_unique_ids = set(city_base_filtered[city_key].unique())
    
    # 2. Outer Scale Loop
    for size in GRID_SIZES:
        input_matrix_path = UNIFIED_DIR / f"merged_metrics_{size}m.parquet"
        
        if not input_matrix_path.exists():
            continue
            
        print(f"\n=======================================================")
        print(f"🔄 ALIGNING INTERSECTION FOR SYSTEM SCALE: {size}m")
        print(f"=======================================================")
        
        grid_metrics_df = pd.read_parquet(input_matrix_path)
        metrics_key = 'city_id'
        
        # Standardize grid tracking keys datatype
        grid_metrics_df[metrics_key] = grid_metrics_df[metrics_key].astype(str).str.strip()
        grid_matrix_unique_ids = set(grid_metrics_df[metrics_key].unique())
        
        # =================================================================
        # 🔬 科学存证：两端不平衡性盘查
        # =================================================================
        # Only print diagnostic metrics on the first resolution run to avoid log pollution
        if size == GRID_SIZES[0]:
            common_cities = grid_matrix_unique_ids.intersection(city_table_unique_ids)
            only_in_grids = grid_matrix_unique_ids - city_table_unique_ids
            only_in_table = city_table_unique_ids - grid_matrix_unique_ids
            
            print(f"📊 [Data Alignment Cross-Check Summary]:")
            print(f"   - Unique cities found in Grid Fishnets : {len(grid_matrix_unique_ids)}")
            print(f"   - Unique cities found in Context Table : {len(city_table_unique_ids)}")
            print(f"   - Successfully Aligned Intersection    : {len(common_cities)} cities")
            
            if only_in_grids:
                print(f"   ⚠️  Dropped from Grids (Missing in Table): {sorted(list(only_in_grids))}")
            if only_in_table:
                print(f"   ℹ️  Unmatched in Table (Missing in Grids): {sorted(list(only_in_table))}")
        # =================================================================
        
        initial_row_count = len(grid_metrics_df)
        
        # 3. Secure Fusion via INNER JOIN 
        # This forcefully trims the rows to only keep records where the key exists in BOTH scopes
        enriched_metrics_df = pd.merge(
            grid_metrics_df,
            city_base_filtered,
            left_on=metrics_key,
            right_on=city_key,
            how='inner' # Trims out unmatched city blocks perfectly
        )
        
        # Clean up link duplicates
        if city_key in enriched_metrics_df.columns and metrics_key != city_key:
            enriched_metrics_df.drop(columns=[city_key], errors='ignore', inplace=True)
            
        # 4. Compute SUHI Index on the clean aligned matrix
        enriched_metrics_df['SUHI'] = enriched_metrics_df['lst_mean'] - enriched_metrics_df['LST_Rural_mean']
        
        # 5. Overwrite the self-contained Parquet matrix back to disk
        enriched_metrics_df.to_parquet(input_matrix_path, index=False, engine='pyarrow')
        
        final_row_count = len(enriched_metrics_df)
        purged_rows = initial_row_count - final_row_count
        
        print(f"💾 SUCCESS: Unified {size}m matrix committed.")
        print(f"   Final Matrix Shape : {final_row_count} rows x {enriched_metrics_df.shape[1]} features.")
        print(f"   Sample Trim Audit  : Safely filtered out {purged_rows} rows from non-intersecting city spaces.")


In [15]:
CITY_BACKGROUND_CSV = "../../data/Unified_City_Rural_Level_Stats_DE.csv"
integrate_city_intersection_pipeline(CITY_BACKGROUND_CSV)

Loading macro city-level matrix: Unified_City_Rural_Level_Stats_DE.csv

🔄 ALIGNING INTERSECTION FOR SYSTEM SCALE: 100m
📊 [Data Alignment Cross-Check Summary]:
   - Unique cities found in Grid Fishnets : 99
   - Unique cities found in Context Table : 97
   - Successfully Aligned Intersection    : 97 cities
   ⚠️  Dropped from Grids (Missing in Table): ['2114', '3004']
💾 SUCCESS: Unified 100m matrix committed.
   Final Matrix Shape : 846443 rows x 51 features.
   Sample Trim Audit  : Safely filtered out 5190 rows from non-intersecting city spaces.

🔄 ALIGNING INTERSECTION FOR SYSTEM SCALE: 250m
💾 SUCCESS: Unified 250m matrix committed.
   Final Matrix Shape : 141172 rows x 51 features.
   Sample Trim Audit  : Safely filtered out 911 rows from non-intersecting city spaces.

🔄 ALIGNING INTERSECTION FOR SYSTEM SCALE: 500m
💾 SUCCESS: Unified 500m matrix committed.
   Final Matrix Shape : 37468 rows x 51 features.
   Sample Trim Audit  : Safely filtered out 247 rows from non-intersecting city

In [2]:
from shapely import wkb

def recover_wkb_geoparquet_pipeline():
    """
    Safely bypasses missing geo-metadata exceptions by loading tables via pandas,
    decoding raw WKB hex bytes into Shapely object arrays, and declaring formal GeoDataFrames.
    """
    GRID_SIZES = [100, 250, 500, 750, 1000]
    UNIFIED_DIR = Path("../../data/unified_scale_matrices").resolve()
    
    # 🚨 科学自检：根据你生成网格时的投影坐标系指定 CRS
    # 如果网格是经纬度，填 "EPSG:4326"；如果是德国常用投影，填 "EPSG:25832" 或 "EPSG:3857"
    TARGET_CRS = "EPSG:25832" 

    for size in GRID_SIZES:
        file_path = UNIFIED_DIR / f"merged_metrics_{size}m.parquet"
        
        if not file_path.exists():
            continue
            
        print(f"\nDecoding binary spatial streams for resolution: {size}m...")
        
        # 1. First layer fallback: Force Pandas to fetch the tabulate grid with raw byte arrays
        df = pd.read_parquet(file_path)
        
        if 'geometry' not in df.columns:
            print(f"   ⚠️ Skipping {file_path.name}: Column 'geometry' missing.")
            continue
            
        # 2. Hardcore parsing block: Awaken hex bytes into explicit Shapely Polygons
        # wkb.loads handles python bytes type natively
        first_sample = df['geometry'].iloc[0]
        
        if isinstance(first_sample, (bytes, str)):
            print("   🧬 WKB footprint detected. Reconstructing spatial topology vectors...")
            df['geometry'] = df['geometry'].apply(
                lambda x: wkb.loads(x) if isinstance(x, bytes) else wkb.loads(x, hex=True)
            )
        
        # 3. Structural Elevation: Re-package the flat dataframe into a spatial vector machine
        grid_gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=TARGET_CRS)
        
        # 4. Standard Re-Commit (Strict GeoParquet Spec enforcement)
        # Using gpd.to_parquet ensures that the 'geo' metadata block is formally written this time
        grid_gdf.to_parquet(file_path, engine='pyarrow')
        
        print(f"💾 SUCCESS: Metadata rewritten cleanly to -> {file_path.name}")
        print(f"   Verification: First row topology is now {type(grid_gdf['geometry'].iloc[0])} -> {str(grid_gdf['geometry'].iloc[0])[:40]}...")

In [5]:
recover_wkb_geoparquet_pipeline()


Decoding binary spatial streams for resolution: 100m...
   🧬 WKB footprint detected. Reconstructing spatial topology vectors...
💾 SUCCESS: Metadata rewritten cleanly to -> merged_metrics_100m.parquet
   Verification: First row topology is now <class 'shapely.geometry.polygon.Polygon'> -> POLYGON ((355323.79933635076 5451076.563...

Decoding binary spatial streams for resolution: 250m...
   🧬 WKB footprint detected. Reconstructing spatial topology vectors...
💾 SUCCESS: Metadata rewritten cleanly to -> merged_metrics_250m.parquet
   Verification: First row topology is now <class 'shapely.geometry.polygon.Polygon'> -> POLYGON ((355273.79933635076 5451076.563...

Decoding binary spatial streams for resolution: 500m...
   🧬 WKB footprint detected. Reconstructing spatial topology vectors...
💾 SUCCESS: Metadata rewritten cleanly to -> merged_metrics_500m.parquet
   Verification: First row topology is now <class 'shapely.geometry.polygon.Polygon'> -> POLYGON ((355523.79933635076 5451076.563..